# Proter — Sistem Pemantauan Lalu Lintas Stereo

Selamat datang! Notebook ini adalah **panduan penggunaan** aplikasi
**Monitor Lalu Lintas Stereo** yang tersimpan di folder proyek ini.

Aplikasi ini bekerja seperti **dua mata manusia**: dua kamera memandang jalan
dari dua posisi yang sedikit berbeda, lalu komputer:

1. **mendeteksi** kendaraan (mobil, motor, bus, truk, sepeda),
2. **melacak** setiap kendaraan dengan nomor ID yang tetap,
3. **mengukur posisi** kendaraan dalam **meter** (bukan sekadar piksel),
4. **menghitung kecepatannya**, dan
5. **menyimpan semua hasil** ke file data yang bisa dibaca program lain.

Kode aplikasi tersimpan dalam dua file Python:

| File | Peran |
|---|---|
| `stereo_traffic_monitor.py` | "Otak" pemrosesan: kalibrasi, deteksi, pelacakan, pengukuran, penyimpanan |
| `stereo_traffic_gui.py` | Jendela aplikasi (tampilan grafis) yang memakai otak tersebut |

> Notebook ini **tidak lagi berisi kode aplikasi**, melainkan penjelasan dan
> contoh yang aman dijalankan: tidak membuka kamera dan tidak mengunduh apa pun
> secara otomatis.

## 1. Mengenal proyek

Bayangkan Anda berdiri di tepi jalan sambil memotret mobil yang lewat dengan
satu kamera. Foto itu cukup untuk tahu *"ada mobil di situ"*, tetapi **tidak
cukup untuk tahu mobil itu 15 meter atau 40 meter dari Anda**.

Kuncinya: **jarak tidak bisa diukur dari satu gambar, tetapi bisa dari dua
gambar yang diambil dari posisi berbeda.** Otak kita melakukan hal yang sama —
perbedaan tampilan antara mata kiri dan mata kanan (*disparity*) dipakai untuk
merasakan kedalaman.

Sistem ini meniru cara itu:

| Kemampuan | Penjelasan |
|---|---|
| Deteksi kendaraan | Model **YOLO** menandai kendaraan pada gambar kiri |
| Pelacakan (tracking) | **ByteTrack** memberi nomor ID yang sama antar-frame, sehingga gerakan tiap kendaraan bisa diikuti |
| Pengukuran jarak | Posisi kendaraan di gambar kiri dibandingkan dengan gambar kanan, lalu dihitung menjadi jarak dalam **meter** |
| Pengukuran kecepatan | Perubahan posisi dari waktu ke waktu, dihaluskan agar tidak "melompat-lompat" |
| Penyimpanan data | Semua hasil ditulis ke **CSV**, video ke **MP4**, dan keterangan ke **JSON** |

**Contoh pemakaian nyata:** memantau kecepatan lalu lintas di depan sekolah,
menghitung jumlah kendaraan per jam, atau mengumpulkan data untuk penelitian.

## 2. Cara kerja sederhana

```
 Kamera kiri          Kamera kanan
     |                     |
     +--------+  +---------+
              v  v
        [1] Rectification
            "meluruskan" gambar agar kedua kamera sejajar sempurna
              |
              v
        [2] Deteksi (YOLO)
            menemukan kendaraan di gambar kiri
              |
              v
        [3] Pelacakan (ByteTrack)
            kendaraan yang sama diberi ID yang sama di setiap frame
              |
              v
        [4] Pencocokan stereo
            kendaraan dicari juga di gambar kanan;
            selisih posisinya (disparity) menunjukkan jaraknya
              |
              v
        [5] Triangulasi
            disparity + jarak antar-kamera  ->  posisi 3D dalam meter
              |
              v
        [6] Estimasi kecepatan
            perubahan posisi dianalisis dari beberapa frame terakhir
              |
              v
   tracking_data.csv + 2 video MP4 + file JSON (kalibrasi & metadata)
```

Tahap 1-6 berjalan **otomatis** setiap frame, baik saat pemantauan langsung
(mode *Live*) maupun saat memproses file video.

## 3. Persiapan

Sebelum menjalankan aplikasi, siapkan:

1. **Python 3.9+** — tkinter (jendela aplikasi) sudah ikut pada instalasi
   Python standar.
2. **Pustaka pendukung** — `opencv-python`, `numpy`, `ultralytics` (YOLO),
   `matplotlib` (opsional, untuk grafik). Cara cepat:

   ```bash
   pip install -r requirements.txt
   ```

3. **Dua kamera** yang dipasang bersebelahan sejajar seperti sepasang mata,
   **atau** dua file video yang terekam bersamaan dari dua sudut pandang.
4. **File kalibrasi** `calibration.json` — dijelaskan pada bagian 4.
5. Bobot model YOLO (`yolov9c.pt`, sekitar 25 MB) akan **diunduh otomatis**
   saat pertama kali aplikasi dijalankan — butuh koneksi internet sekali saja.

Sel berikut memeriksa kesiapan komputer Anda. **Sel ini aman dijalankan
kapan saja** — tidak membuka kamera dan tidak mengunduh apa-apa.

In [ ]:
# Cek kesiapan lingkungan proyek
import importlib.util
import sys
from pathlib import Path

print(f"Python       : {sys.version.split()[0]}")
print()

for nama in ("numpy", "cv2", "ultralytics", "matplotlib", "tkinter"):
    ada = importlib.util.find_spec(nama) is not None
    print(f"  {nama:<12}: {'OK' if ada else 'BELUM TERPASANG'}")

print()
for f in ("stereo_traffic_monitor.py", "stereo_traffic_gui.py",
          "calibration.json", "requirements.txt"):
    print(f"  {f:<30}: {'ada' if Path(f).exists() else 'TIDAK ADA (opsional)' if f == 'calibration.json' else 'TIDAK ADA'}")

print()
if (importlib.util.find_spec("cv2") is None
        or importlib.util.find_spec("numpy") is None):
    print("Pustaka inti belum lengkap. Jalankan di terminal:")
    print("    pip install -r requirements.txt")
else:
    print("Pustaka inti lengkap - notebook siap dipakai.")

## 4. Kalibrasi kamera (dilakukan sekali)

Kalibrasi adalah proses mengenali "karakter" pasangan kamera kita: seberapa
melengkung lensanya, ke mana kedua kamera menghadap, dan **seberapa jauh
jarak antara kedua kamera** (*baseline*). Tanpa kalibrasi, hasil pengukuran
hanyalah piksel — setelah kalibrasi, hasilnya menjadi **meter**.

**Cara kalibrasi (30-60 menit):**

1. Cetak **papan catur** 9x6 kotak (ukuran kotak 2,5 cm) dan tempel di alas
   yang rata/keras.
2. Dengan **kedua kamera tetap terpasang kuat** (tidak boleh bergeser lagi
   setelah ini!), foto papan catur dari 10-15 posisi dan kemiringan berbeda.
   Kedua kamera harus memotret **pada saat yang bersamaan**.
3. Simpan foto kiri di folder `foto/kiri` dan foto kanan di folder `foto/kanan`
   (nama file berpasangan dan terurut sama, misalnya `img01.jpg`, `img02.jpg`, ...).
4. Jalankan perintah kalibrasi di terminal (lihat sel berikut).
5. Periksa hasilnya: nilai **RMS error** idealnya di bawah sekitar **1,0 piksel**.

> **Penting:** setelah kalibrasi, jangan menggeser atau memutar kamera.
> Jika posisi kamera berubah, kalibrasi harus diulang.

In [ ]:
# Perintah kalibrasi - dibaca saja, TIDAK dijalankan dari notebook.
print("Langkah 1-3: foto papan catur 9x6 dari dua kamera secara bersamaan,")
print("             simpan di folder foto/kiri dan foto/kanan.")
print()
print("Langkah 4 : jalankan di terminal:")
print()
print("    python stereo_traffic_monitor.py calibrate \\")
print("        --left-dir foto/kiri --right-dir foto/kanan \\")
print("        --board 9x6 --square-m 0.025 --out calibration.json")
print()
print("Langkah 5 : periksa pesan 'RMS error' - idealnya di bawah sekitar 1.0 piksel.")

## 5. Menjalankan aplikasi

Aplikasi punya dua mode utama yang bisa dipilih di **tab kanan** jendela:

| Mode | Fungsi |
|---|---|
| **Live** | Memantau langsung: pilih sumber kiri/kanan (nomor kamera atau file video), file kalibrasi, lalu tekan **Start**. Kotak hijau/biru, ID, jarak, dan kecepatan muncul di atas kendaraan secara waktu-nyata. |
| **Analysis** | Membuka hasil rekaman untuk dianalisis: play/pause, lompat ke frame mana pun, maju 1 frame, ganti kecepatan putar, tampilkan kiri/kanan/berdampingan. |

Cara paling mudah adalah membuka jendela aplikasi:

```bash
python stereo_traffic_monitor.py gui
```

Untuk pemrosesan **tanpa jendela** (cocok untuk server atau pengumpulan data
jangka panjang), gunakan perintah `run` — lihat sel berikut.

In [ ]:
# Ringkasan perintah aplikasi - dibaca saja, TIDAK dijalankan dari notebook.
# (Membuka jendela GUI atau kamera dari dalam notebook tidak disarankan.)
print("""
Mode GUI (jendela aplikasi, tab Live + tab Analysis):
    python stereo_traffic_monitor.py gui

Mode headless - memproses dua file video tanpa tampilan:
    python stereo_traffic_monitor.py run --left video_kiri.mp4 --right video_kanan.mp4 \\
        --calibration calibration.json --out traffic_monitoring

Mode headless - memakai dua kamera langsung (indeks 0 dan 1):
    python stereo_traffic_monitor.py run --left 0 --right 1 \\
        --calibration calibration.json --out traffic_monitoring

Menambahkan tampilan pratinjau: tambahkan opsi  --preview
Menonaktifkan penyimpanan video: tambahkan opsi  --no-video

Mode overlay - membuat video berisi ID + kecepatan dari sebuah rekaman:
    python stereo_traffic_monitor.py overlay recording_001 --camera left --unit km/h
""")

## 6. Membaca hasil: isi folder rekaman

Setiap sesi rekaman menghasilkan satu folder:

```
traffic_monitoring/
└── recording_001/
    ├── left_camera.mp4          video kiri yang sudah "diluruskan" (rectified)
    ├── right_camera.mp4         video kanan yang sudah "diluruskan"
    ├── tracking_data.csv        PUSAT DATA: satu baris per kendaraan per frame
    ├── calibration.json         hasil kalibrasi + definisi sistem koordinat
    └── recording_metadata.json  waktu mulai, fps, resolusi, info sinkronisasi
```

### Arti kolom `tracking_data.csv`

| Kelompok kolom | Arti |
|---|---|
| `timestamp_ns` | Waktu tangkap dalam **nanodetik** sejak 1 Jan 1970 |
| `frame_id` | Nomor frame, **mulai dari 1** — kunci untuk menyandingkan baris CSV dengan video |
| `object_id` | Nomor ID kendaraan; berlaku tetap **di dalam satu rekaman** |
| `class`, `confidence` | Jenis kendaraan dan tingkat keyakinan **deteksi YOLO** |
| `world_x_m`, `world_y_m`, `world_z_m` | Posisi 3D dalam **meter**: X ke kanan, Y ke bawah, Z ke depan (kedalaman) |
| `velocity_x_ms`, `velocity_y_ms`, `velocity_z_ms`, `speed_ms` | Kecepatan dalam m/s — berupa **vektor** (tahu arah) dan skalar |
| `left_u`, `left_v` | Posisi piksel pusat kendaraan di gambar kiri |
| `right_u`, `right_v` | Posisi yang sama di gambar kanan |
| `bbox_x`, `bbox_y`, `bbox_w`, `bbox_h` | Kotak pembatas kendaraan (bounding box) di gambar kiri |

> **Kenapa desain ini berguna?** Program lain (Excel, dasbor web, aplikasi
> Android, dll.) cukup membaca CSV — tanpa perlu pustaka computer vision, tanpa
> perlu tahu cara kerja YOLO. Sel berikut meniru cara program lain membaca data.

In [ ]:
# Membaca hasil pengukuran TANPA pustaka computer vision.
# Hanya memakai csv bawaan Python - sama seperti yang akan dilakukan program lain.
import csv
from pathlib import Path

kandidat = [Path("tracking_data.csv"),
            *sorted(Path(".").glob("recording_*/tracking_data.csv"))]
csv_path = next((p for p in kandidat if p.exists()), None)

if csv_path is None:
    print("Belum ada tracking_data.csv di folder proyek ini.")
    print("Jalankan dulu mode Live / perintah 'run' (bagian 5), lalu jalankan sel ini lagi.")
else:
    print(f"Membaca: {csv_path}")
    with open(csv_path, newline="") as f:
        baris = list(csv.DictReader(f))
    kendaraan = sorted({b["object_id"] for b in baris})
    print(f"Jumlah baris pengamatan : {len(baris)}")
    print(f"Kendaraan terlacak      : {len(kendaraan)} (ID: {', '.join(kendaraan[:8])}" +
          ('...' if len(kendaraan) > 8 else '') + ')')
    print()
    print("Contoh 3 baris pertama:")
    for b in baris[:3]:
        print(f"  frame {b['frame_id']:>4}  #{b['object_id']:<4} {b['class']:<10} "
              f"posisi ({float(b['world_x_m']):+6.1f}, {float(b['world_y_m']):+5.1f}, "
              f"{float(b['world_z_m']):5.1f}) m   {float(b['speed_ms']) * 3.6:5.1f} km/h")

## 7. Contoh analisis sederhana

Karena data sudah berupa angka meter dan m/s, analisis traffic bisa dilakukan
langsung dari CSV. Contoh di bawah menggambar **grafik kecepatan** setiap
kendaraan dari waktu ke waktu — dari sini terlihat kendaraan yang melaju kencang
atau menginjak rem.

Contoh analisis lain yang bisa dikembangkan:
- jumlah kendaraan per menit (arahnya bisa ditentukan dari tanda `velocity_x_ms`),
- kecepatan rata-rata / persentil ke-85 per jenis kendaraan,
- jarak antar kendaraan (*headway*) di satu lajur,
- kepadatan lalu lintas per segmen jarak.

In [ ]:
# Grafik kecepatan per kendaraan (butuh matplotlib dan file tracking_data.csv).
import csv
from collections import defaultdict
from pathlib import Path

kandidat = [Path("tracking_data.csv"),
            *sorted(Path(".").glob("recording_*/tracking_data.csv"))]
csv_path = next((p for p in kandidat if p.exists()), None)

if csv_path is None:
    print("Belum ada tracking_data.csv - jalankan dulu mode Live / perintah 'run' (bagian 5).")
else:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib belum terpasang. Pasang dulu:  pip install matplotlib")
    else:
        riwayat = defaultdict(list)
        with open(csv_path, newline="") as f:
            baris = list(csv.DictReader(f))
        t0 = min(int(b["timestamp_ns"]) for b in baris)
        for b in baris:
            t = (int(b["timestamp_ns"]) - t0) / 1e9          # detik sejak awal rekaman
            riwayat[b["object_id"]].append((t, float(b["speed_ms"]) * 3.6))  # km/h

        plt.figure(figsize=(9, 4))
        for oid, titik in sorted(riwayat.items())[:6]:        # maksimal 6 kendaraan pertama
            t, v = zip(*titik)
            plt.plot(t, v, marker=".", linewidth=1, label=f"#{oid}")
        plt.xlabel("waktu (detik)")
        plt.ylabel("kecepatan (km/h)")
        plt.title(f"Kecepatan kendaraan - {csv_path}")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

## 8. Memutar ulang rekaman (mode Analysis)

Mode **Analysis** dipakai untuk memeriksa hasil rekaman tanpa menghitung ulang
apa pun — overlay yang tampil dibaca langsung dari `tracking_data.csv`.

**Cara membuka rekaman:**

1. Jalankan aplikasi: `python stereo_traffic_monitor.py gui`
2. Pindah ke tab **Analysis** (kanan atas).
3. Tekan **"Open recording folder..."** lalu pilih folder `recording_001`
   (atau **"Open videos + CSV manually..."** untuk memilih file satu per satu).

**Kendali pemutaran:**

| Kendali | Fungsi |
|---|---|
| `Spasi` | play / pause |
| `←` / `→` | mundur / maju **satu frame** |
| Slider | lompat ke frame mana pun |
| `0.25x - 4x, max` | kecepatan putar |
| `Loop` | mengulang dari awal saat selesai |
| Klik baris tabel | menyorot satu kendaraan (info detail tampil di videonya) |

Tanpa GUI pun bisa: perintah `overlay` (bagian 5) membuat video berisi ID,
kecepatan, dan lintasan langsung dari folder rekaman.

## 9. Batasan sistem (penting dipahami)

| Batasan | Penjelasan | Dampak & saran |
|---|---|---|
| Kamera USB tidak tersinkron hardware | Kedua kamera dibaca bergantian; dua frame bisa berselisih hingga ~1 frame | Kedalaman kendaraan yang sangat cepat bisa meleset. Untuk akurasi tinggi, gunakan kamera dengan sinkronisasi hardware |
| `confidence` bukan skor akurasi 3D | Angka itu hanya keyakinan **deteksi YOLO** | Jangan anggap kendaraan ber-`confidence` tinggi pasti diukur tepat |
| Bounding box kanan bukan deteksi independen | Kotak di gambar kanan adalah kotak kiri yang digeser sebesar disparity | Wajar jika kotak kanan terlihat kurang pas |
| Akurasi bergantung kalibrasi | Baseline salah sedikit -> jarak salah sebesar-besarnya | Kalibrasi ulang setiap kali kamera digeser; validasi dengan pengukuran lapangan |
| Objek sulit dipasangkan | Oklusi, gelap, atau tekstur lemah bisa gagal dicocokkan antar kamera | Kendaraan tersebut **dibuang** dari hasil 3D (bukan ditebak) — data tetap jujur |
| `object_id` bisa berganti | Pelacak bisa "kehilangan" kendaraan lalu memberi ID baru | Setelah tersambung kembali, ID lama & baru adalah kendaraan yang sama |

> **Catatan jujur untuk penelitian:** angka kecepatan dari sistem ini **belum
> divalidasi** terhadap alat ukur pembanding (radar/GPS). Sebelum dipakai untuk
> kesimpulan ilmiah, bandingkan hasilnya dengan acuan di lapangan.

## 10. Pemecahan masalah umum

| Gejala | Kemungkinan penyebab | Solusi |
|---|---|---|
| `tkinter is not available` | Python tanpa tkinter | Windows/macOS: instal ulang Python dari python.org (centang tcl/tk). Debian/Ubuntu: `sudo apt install python3-tk` |
| `Cannot open left source '0'` | Kamera tidak terbuka / dipakai aplikasi lain | Tutup aplikasi lain (Zoom, dsb.), cek indeks kamera lewat tombol **Scan cameras** |
| `Cannot open ... source 'video.mp4'` | File tidak ada / format tak didukung | Pastikan path benar; gunakan mp4/avi/mkv/mov |
| `Calibration file not found` | `calibration.json` belum dibuat | Lakukan kalibrasi dulu (bagian 4) |
| Jauh lebih banyak `[skip]` saat kalibrasi | Papan catur tak terbaca | Perbaiki pencahayaan, pastikan papan rata dan terlihat utuh di kedua kamera |
| Unduhan YOLO gagal / lambat | Koneksi internet | Bobot `yolov9c.pt` bisa diunduh manual dari GitHub Ultralytics, letakkan di folder proyek |
| Overlay tidak sinkron dengan video | File CSV dan video dari rekaman berbeda | Selalu buka rekaman lewat **folder** (Analysis) agar CSV dan video berasal dari sesi yang sama |
| Semua kecepatan 0 | Baru 1-2 frame terlacak | Kecepatan dihitung setelah ada 3 pengamatan; tunggu beberapa frame |
| Notebook error `ModuleNotFoundError: cv2` | Pustaka belum terpasang di Python yang dipakai Jupyter | Jalankan `pip install -r requirements.txt` di lingkungan kernel notebook |

## Struktur proyek & langkah berikutnya

```
proter/
├── proter.ipynb                  <- notebook ini (panduan)
├── stereo_traffic_monitor.py     inti pemrosesan + perintah CLI
├── stereo_traffic_gui.py         jendela aplikasi (tab Live + Analysis)
├── requirements.txt              daftar pustaka
├── tests/test_tracking_data.py   pengujian otomatis lapisan data
├── proter_backup_original.ipynb  salinan notebook asli (arsip)
└── traffic_monitoring/           hasil rekaman (muncul setelah sesi pertama)
```

**Uji otomatis** (tanpa kamera, memeriksa geometri, kecepatan, dan round-trip CSV):

```bash
python tests/test_tracking_data.py
```

**Ide pengembangan berikutnya** (sengaja belum diimplementasikan, agar basis
data tetap sederhana): ekspor ke SQLite/Parquet ketika data membesar, arah
lajur otomatis, dasbor web, atau sinkronisasi hardware kamera.

*Selamat mengukur!*